<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/01_consolidar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 01_consolidar
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se enfoca en la integración de las distintas fuentes de datos en un formato cohesivo y estructurado. Aquí transformamos múltiples conjuntos de datos en una base unificada que servirá para los análisis posteriores.

**Propósito:** Crear una vista unificada y coherente de todos los datos recolectados, facilitando su posterior procesamiento y análisis.

**Tareas habituales:**
- Renombrar archivos
- Unión vertical de archivos complementarios (`union`)
- Combinar archivos (`joins`: inner, left, right, full outer)
- Estandarización inicial de formatos de columnas
- Verificación de consistencia en las uniones
- Validación de cardinalidad en las relaciones
- Gestión de duplicados producto de las uniones

In [2]:
from google.colab import drive
import os, pandas as pd

drive.mount('/content/drive')

RAW_PATH     = "/content/drive/MyDrive/proyecto_oro/data/raw/"
LANDING_PATH = "/content/drive/MyDrive/proyecto_oro/data/landing/"
os.makedirs(LANDING_PATH, exist_ok=True)

dfs = []

for archivo in os.listdir(RAW_PATH):
    if not archivo.endswith('.csv'):
        continue

    nombre = archivo.replace('.csv', '')
    df = pd.read_csv(RAW_PATH + archivo, index_col=0, parse_dates=True)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = [f"{nombre}_{col}" for col in df.columns]
    dfs.append(df)
    print(f"✓ {archivo}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

df_landing = pd.concat(dfs, axis=1, join='outer')
df_landing = df_landing.sort_index()
df_landing = df_landing.ffill(limit=3)
df_landing = df_landing[df_landing.index.dayofweek < 5]
df_landing.index.name = 'DATE'
df_landing = df_landing.reset_index()
df_landing['DATE'] = df_landing['DATE'].astype(str)

ruta_salida = LANDING_PATH + "consolidado_oro_dxy.csv"
df_landing.to_csv(ruta_salida, index=False)

print(f"\n✅ Landing listo")
print(f"📊 {df_landing.shape[0]:,} filas × {df_landing.shape[1]} columnas")
print(f"📅 {df_landing['DATE'].min()} → {df_landing['DATE'].max()}")
df_landing.head()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✓ gvz.csv: 1,378 filas × 5 columnas
✓ cobre.csv: 1,381 filas × 5 columnas
✓ eur_usd.csv: 1,428 filas × 5 columnas
✓ bono_10y.csv: 1,378 filas × 5 columnas
✓ plata_xagusd.csv: 1,378 filas × 5 columnas
✓ dxy.csv: 1,380 filas × 5 columnas
✓ vix.csv: 1,379 filas × 5 columnas
✓ wti_crudo.csv: 1,380 filas × 5 columnas
✓ oro_xauusd.csv: 1,378 filas × 5 columnas
✓ yield_real_10y.csv: 1,434 filas × 1 columnas
✓ breakeven_inflacion.csv: 1,434 filas × 1 columnas
✓ spread_curva_10y2y.csv: 1,434 filas × 1 columnas
✓ bono_2y.csv: 1,434 filas × 1 columnas

✅ Landing listo
📊 1,434 filas × 50 columnas
📅 2021-01-01 → 2026-07-01


,DATE,gvz_Close,gvz_High,gvz_Low,gvz_Open,gvz_Volume,cobre_Close,cobre_High,cobre_Low,cobre_Open,...,wti_crudo_Volume,oro_xauusd_Close,oro_xauusd_High,oro_xauusd_Low,oro_xauusd_Open,oro_xauusd_Volume,yield_real_10y_yield_real_10y,breakeven_inflacion_breakeven_inflacion,spread_curva_10y2y_spread_curva_10y2y,bono_2y_bono_2y
0,2021-01-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-04,21.860001,22.360001,21.240000,22.090000,0.0,3.5530,3.6005,3.5455,3.5590,...,528525.0,182.330002,182.399994,180.960007,181.970001,14331400.0,-1.08,2.01,0.82,0.11
2,2021-01-05,22.139999,22.260000,20.920000,21.110001,0.0,3.6405,3.6555,3.5900,3.5900,...,643191.0,182.869995,183.210007,181.820007,182.869995,12718800.0,-1.07,2.03,0.83,0.13
3,2021-01-06,21.700001,21.809999,20.590000,20.860001,0.0,3.6500,3.6905,3.6400,3.6505,...,509365.0,179.899994,181.580002,178.240005,181.490005,18453500.0,-1.02,2.06,0.90,0.14
4,2021-01-07,20.660000,21.420000,20.389999,21.240000,0.0,3.6955,3.7080,3.6430,3.6590,...,369292.0,179.479996,179.919998,178.839996,179.690002,7110200.0,-1.01,2.09,0.94,0.14
